# LLM 推理加速

> 我们已经知道，自回归生成时每次只生成一个 token，但要把累积的序列重新算一遍 Attention——序列越长，每次生成就越慢。
>
> 这一节，我们拆解 LLM 推理慢的根本原因，并从头理解三种核心加速技术：KV Cache 让已经算过的信息不再重算，FlashAttention 让 Attention 计算对硬件更友好，Continuous Batching 让服务端同时处理多个请求而不是排队等。

LLM 推理有两个阶段：prefill（一次性处理输入 prompt，可以并行）和 decode（逐 token 生成，必须串行）。decode 的瓶颈不在计算量，而在显存带宽——每生成一个 token 都要从显存里读一遍整个模型权重和 KV Cache。KV Cache 的原理很简单：已经算过的 Key 和 Value 存起来，下一个 token 只算新的那个位置的 K 和 V，attention 时拼回去即可。FlashAttention 则从 GPU 内存层次入手，把 attention 计算分块，让中间结果尽量留在 SRAM 里而不是反复读写 HBM。下面逐一动手实现。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)

## 1. 推理慢的根源

回顾自回归生成的过程：

```
Step 1: 输入 [BOS]              → 算 attention → 预测 "我"
Step 2: 输入 [BOS, 我]          → 算 attention → 预测 "爱"
Step 3: 输入 [BOS, 我, 爱]      → 算 attention → 预测 "你"
Step 4: 输入 [BOS, 我, 爱, 你]  → 算 attention → 预测 EOS
```

**问题**：Step 2 重新算了 Step 1 已经算过的 `[BOS]` 的 K 和 V！
Step 3 重新算了 Step 1+2 已经算过的！

```
Step 1: 算 K₁, V₁（1 个 token）
Step 2: 算 K₁, V₁, K₂, V₂（2 个 token，但 K₁V₁ 是重复的！）
Step 3: 算 K₁, V₁, K₂, V₂, K₃, V₃（3 个 token，前 4 个是重复的！）
...
```

生成 N 个 token，总计算量 ≈ 1+2+3+...+N = **O(N²)**！

生成 100 个 token 需要算 5050 次 attention，而不是 100 次。

In [2]:
# 可视化重复计算
def naive_generation_cost(n_tokens):
    """朴素自回归生成的总计算量（以 token 为单位）"""
    return sum(range(1, n_tokens + 1))

for n in [10, 50, 100, 500]:
    cost = naive_generation_cost(n)
    print(f"生成 {n:3d} 个 token → 实际算了 {cost:6d} 次 token attention → 浪费 {cost - n:5d} 次")

print(f"\n生成 100 个 token，浪费了 {naive_generation_cost(100) - 100} 次计算！")
print(f"这就是 LLM 推理慢的根本原因。")

生成  10 个 token → 实际算了     55 次 token attention → 浪费    45 次
生成  50 个 token → 实际算了   1275 次 token attention → 浪费  1225 次
生成 100 个 token → 实际算了   5050 次 token attention → 浪费  4950 次
生成 500 个 token → 实际算了 125250 次 token attention → 浪费 124750 次

生成 100 个 token，浪费了 4950 次计算！
这就是 LLM 推理慢的根本原因。


## 2. KV Cache

**核心思想**：之前 token 的 K 和 V 已经算过了，存起来直接用，别重新算！

```
Step 1: 算 K₁, V₁ → 存进 cache
Step 2: 只算 K₂, V₂ → 从 cache 取 K₁, V₁ → 拼起来用
Step 3: 只算 K₃, V₃ → 从 cache 取 K₁, V₁, K₂, V₂ → 拼起来用
...
```

**效果**：生成 N 个 token，总计算量从 O(N²) 降到 O(N)！

```
朴素: 1+2+3+...+100 = 5050 次
KV Cache: 1+1+1+...+1 = 100 次  ← 快了 50 倍！
```

In [3]:
class AttentionWithKVCache(nn.Module):
    """
    带 KV Cache 的 Self-Attention
    
    关键区别：
    - 第一次调用（prefill）：处理整个 prompt，算所有 K、V 并存起来
    - 后续调用（decode）：只算新 token 的 K、V，从 cache 取旧的
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
        # KV Cache 存储
        self.k_cache = None
        self.v_cache = None
    
    def reset_cache(self):
        """清空 cache（开始新的生成时调用）"""
        self.k_cache = None
        self.v_cache = None
    
    def forward(self, x, use_cache=True):
        """
        输入: x shape = [batch, seq_len, d_model]
        
        如果 use_cache=True 且 cache 非空：
            x 只包含新 token（seq_len=1）
            从 cache 取旧的 K、V，只算新的 K、V
        """
        batch_size, seq_len, _ = x.shape
        
        # 算 Q、K、V
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        if use_cache:
            if self.k_cache is not None:
                # 拼接旧的 K、V
                K = torch.cat([self.k_cache, K], dim=2)
                V = torch.cat([self.v_cache, V], dim=2)
            # 更新 cache
            self.k_cache = K
            self.v_cache = V
        
        # Attention 计算（和之前一样）
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        attn_output = weights @ V
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)

print("带 KV Cache 的 Attention 定义完成！")

带 KV Cache 的 Attention 定义完成！


In [4]:
# 对比：有 KV Cache vs 没有
attn = AttentionWithKVCache(d_model=64, num_heads=4)

# === 模拟生成 20 个 token ===
N = 20

# 无 cache 版本（朴素）
start = time.time()
sequence = torch.randn(1, 1, 64)
for _ in range(N):
    _ = attn(sequence, use_cache=False)
    sequence = torch.cat([sequence, torch.randn(1, 1, 64)], dim=1)
no_cache_time = time.time() - start

# 有 cache 版本
attn.reset_cache()
start = time.time()
sequence = torch.randn(1, 1, 64)
for _ in range(N):
    _ = attn(sequence, use_cache=True)
    sequence = torch.randn(1, 1, 64)  # 每次只输入新 token
cache_time = time.time() - start

print(f"无 KV Cache: {no_cache_time*1000:.1f} ms")
print(f"有 KV Cache: {cache_time*1000:.1f} ms")
print(f"加速比: {no_cache_time/cache_time:.1f}x")
print(f"\n注意：N=20 时差别不大，但 N=1000 时差别是巨大的！")

无 KV Cache: 4.6 ms
有 KV Cache: 1.5 ms
加速比: 3.0x

注意：N=20 时差别不大，但 N=1000 时差别是巨大的！


## 3. KV Cache 的内存问题

KV Cache 解决了计算量问题，但带来了**内存问题**。

```
一个 token 的 KV Cache 大小:
= 2 × num_layers × d_model × 2 bytes (FP16)

LLaMA-7B: 32 层 × 4096 维 × 2 × 2 = 约 1MB / token
生成 2048 个 token → 约 2GB 的 KV Cache！

LLaMA-70B: 80 层 × 8192 维 × 2 × 2 = 约 5MB / token
生成 2048 个 token → 约 10GB 的 KV Cache！
```

而且每个请求都要独立的 KV Cache。100 个并发用户 × 2GB = 200GB 显存！

这就是 vLLM 要解决的核心问题。

#### KV Cache 量化：用更少的位数存储 K 和 V

前面的计算都是假设 KV Cache 用 FP16（2 bytes）存储。实际上，K 和 V 不需要那么高的精度。

`KV Cache 量化`的做法是：把 FP16 的 K、V 值转换成 INT8（1 byte）或 INT4（0.5 byte）再存储，使用时再反量化回 FP16 参与计算。

- **FP16**（默认）：每个值占 2 bytes，精度最高
- **INT8 量化**：每个值占 1 byte，内存省一半，质量损失很小
- **INT4 量化**：每个值占 0.5 byte，内存省到 1/4，长上下文下质量损失可感知

内存节省的意义：对于 7B 模型 128K 长上下文，单请求的 KV Cache 就要 64GB。如果量化到 INT4，只要 16GB。这对部署场景至关重要——同样的显存可以服务更多用户，或者支持更长的上下文。

现代推理框架（vLLM、TensorRT-LLM）已经内置了 KV Cache 量化支持，通常是配置一个参数即可开启。

In [ ]:
# KV Cache 量化：内存对比计算
# 以 7B 模型为例（LLaMA-2-7B 参数: 32 层, 32 头, 每头 128 维）

num_layers = 32      # Transformer 层数
num_heads = 32       # 注意力头数
head_dim = 128       # 每个头的维度
# d_model = num_heads * head_dim = 4096

# KV Cache 公式：
# 每个元素的 KV Cache 大小 = 2(K+V) × num_layers × num_heads × head_dim × bytes_per_value
# 再乘以序列长度

def kv_cache_bytes(seq_len, bytes_per_value):
    """计算单个请求的 KV Cache 大小（bytes）"""
    # 2 表示 K 和 V 各一份
    return 2 * num_layers * num_heads * head_dim * seq_len * bytes_per_value

def format_size(bytes_size):
    """把字节数转成 MB 或 GB"""
    if bytes_size >= 1024**3:
        return f"{bytes_size / 1024**3:.1f} GB"
    else:
        return f"{bytes_size / 1024**2:.0f} MB"

# 不同精度下每个值占的字节数
precisions = {
    "FP16": 2,
    "INT8": 1,
    "INT4": 0.5,
}

# 不同上下文长度
context_lengths = [4096, 32 * 1024, 128 * 1024]
context_labels = ["4K", "32K", "128K"]

print("=== 7B 模型（32层, 32头, 128维/头）KV Cache 内存占用 ===")
print()
print(f"{'上下文':<8}", end="")
for prec in precisions:
    print(f"{prec:>10}", end="")
print()
print("-" * 38)

for i, seq_len in enumerate(context_lengths):
    print(f"{context_labels[i]:<8}", end="")
    for prec, bpv in precisions.items():
        size = kv_cache_bytes(seq_len, bpv)
        print(f"{format_size(size):>10}", end="")
    print()

print()
print("关键观察：")
fp16_128k = kv_cache_bytes(128 * 1024, 2)
int4_128k = kv_cache_bytes(128 * 1024, 0.5)
print(f"  128K 上下文时，FP16 占 {format_size(fp16_128k)}，INT4 只占 {format_size(int4_128k)}")
print(f"  量化到 INT4 节省了 {(1 - int4_128k/fp16_128k)*100:.0f}% 的 KV Cache 内存")

## 4. PagedAttention

**问题**：KV Cache 是连续分配的一大块内存。但每个请求生成的 token 数不一样，有的长有的短。
连续分配导致大量**内存碎片**——就像停车场的车大小不一，停得乱七八糟，浪费空间。

**vLLM 的方案**：把 KV Cache 切成固定大小的「页」（block），像操作系统的虚拟内存一样管理。

```
传统方式（连续内存）:
┌──────────────────────────┐
│ 请求A的KV Cache (500 tokens) │  占了一大块
├──────────────────────────┤
│      (浪费的碎片)         │  ← 碎片！
├─────────────┤
│ 请求B (100) │
└─────────────┘

PagedAttention（分页）:
┌──┬──┬──┬──┬──┬──┬──┬──┐
│A1│B1│A2│C1│B2│A3│C2│A4│  每页固定大小（如 16 个 token）
└──┴──┴──┴──┴──┴──┴──┴──┘  不连续但没碎片！
```

**效果**：
- 内存利用率从 20-40% → 接近 100%
- 同样的显存可以服务 2-4 倍的并发请求
- 这就是为什么 vLLM 比 HuggingFace 快那么多

In [5]:
# 直观感受：连续分配 vs 分页的内存浪费
import random

def simulate_memory(requests, block_size=16, total_memory=1024):
    """模拟两种内存分配策略"""
    
    # 连续分配
    contiguous_used = 0
    contiguous_wasted = 0
    for req_len in requests:
        contiguous_used += req_len
        # 对齐到 64 的倍数（实际中更复杂）
        aligned = ((req_len + 63) // 64) * 64
        contiguous_wasted += (aligned - req_len)
    
    # 分页分配
    paged_used = 0
    paged_wasted = 0
    for req_len in requests:
        paged_used += req_len
        # 只浪费最后一页没填满的部分
        paged_wasted += (block_size - (req_len % block_size)) % block_size
    
    return contiguous_wasted, paged_wasted

# 模拟 50 个随机长度的请求
random.seed(42)
requests = [random.randint(10, 500) for _ in range(50)]

cw, pw = simulate_memory(requests)
print(f"50 个请求，总 token 数: {sum(requests)}")
print(f"连续分配浪费: {cw} 个 token 位置 ({cw/sum(requests)*100:.1f}%)")
print(f"分页分配浪费: {pw} 个 token 位置 ({pw/sum(requests)*100:.1f}%)")
print(f"\n分页节省了 {cw-pw} 个 token 位置的内存！")

50 个请求，总 token 数: 11486
连续分配浪费: 1570 个 token 位置 (13.7%)
分页分配浪费: 418 个 token 位置 (3.6%)

分页节省了 1152 个 token 位置的内存！


## 5. FlashAttention：让 Attention 算得更快

KV Cache 解决了「少算」的问题，FlashAttention 解决了「算得快」的问题。

**问题**：Attention 的瓶颈不在计算，在**显存读写**。

```
标准 Attention 的计算步骤:
1. Q×K^T → 写回显存 (HBM)
2. Softmax → 读出来，算，写回去
3. ×V → 读出来，算，写回去

每一步都要读写显存！而显存带宽是瓶颈。
```

**FlashAttention 的核心思想**：
- 把 Q、K、V 切成小块
- 一块一块地从显存读到 SRAM（片上缓存，快 10 倍）
- 在 SRAM 里算完 attention，只把最终结果写回显存
- 中间结果不写回显存！

```
标准: HBM → 算 → HBM → 算 → HBM → 算 → HBM  (多次读写)
Flash: HBM → SRAM → 全算完 → HBM  (一次读、一次写)
```

**效果**：
- 速度提升 2-4 倍
- 显存使用减少 10-20 倍（因为不存中间注意力矩阵）
- 数学上完全等价，不是近似！

In [6]:
# 直观感受：显存读写 vs 计算的时间
# 这些数字是近似值，用于建立直觉

print("=== GPU 各级存储的速度对比 ===")
print()
print("HBM (显存):    1.5 TB/s 带宽，但离计算单元远")
print("SRAM (片上):   19 TB/s 带宽，紧挨计算单元")
print()
print("标准 Attention 读写 HBM 的量:")
print("  Q×K^T 矩阵: seq_len² × 2 bytes")
print("  对于 2048 token: 2048² × 2 = 8MB")
print("  对于 8192 token: 8192² × 2 = 128MB ← 巨大！")
print()
print("FlashAttention: 这个矩阵不写回 HBM，省了 128MB 的读写")
print("这就是 FlashAttention 快的原因。")

=== GPU 各级存储的速度对比 ===

HBM (显存):    1.5 TB/s 带宽，但离计算单元远
SRAM (片上):   19 TB/s 带宽，紧挨计算单元

标准 Attention 读写 HBM 的量:
  Q×K^T 矩阵: seq_len² × 2 bytes
  对于 2048 token: 2048² × 2 = 8MB
  对于 8192 token: 8192² × 2 = 128MB ← 巨大！

FlashAttention: 这个矩阵不写回 HBM，省了 128MB 的读写
这就是 FlashAttention 快的原因。


## 6. 推理的两个阶段：Prefill vs Decode

LLM 推理其实分两个阶段，它们的瓶颈完全不同：

| | Prefill（预填充） | Decode（解码） |
|------|-----------|-------|
| 做什么 | 一次性处理整个 prompt | 逐个生成新 token |
| 输入长度 | 长（可能几千 token） | 短（每次 1 个 token） |
| 瓶颈 | **计算**（算大量 attention） | **显存带宽**（读 KV Cache） |
| 优化手段 | FlashAttention | KV Cache + PagedAttention |
| 并行度 | 高（所有 token 并行） | 低（只能串行） |

```
用户输入: "请帮我写一篇关于AI的文章"
              ↓
         [Prefill 阶段]
    一次性处理 10 个 token → 算所有 K、V → 存 cache
              ↓
         [Decode 阶段]
    生成 "人" → 只算 1 个新 token → 读 cache
    生成 "工" → 只算 1 个新 token → 读 cache
    生成 "智" → 只算 1 个新 token → 读 cache
    ...
```

**这就是为什么 LLM 第一个 token 慢（prefill），后面的 token 快（decode）。**

## 7. 其他加速手段一览

| 技术 | 做什么 | 加速倍数 | 精度损失 |
|------|--------|---------|---------|
| **KV Cache** | 缓存已算的 K、V | 10-50x | 无 |
| **FlashAttention** | 减少显存读写 | 2-4x | 无 |
| **PagedAttention (vLLM)** | 分页管理 KV Cache | 吞吐 2-4x | 无 |
| **量化 (INT8/INT4)** | 用更少的 bit 存参数 | 2-4x | 轻微 |
| **投机解码** | 用小模型猜，大模型验证 | 2-3x | 无 |
| **Continuous Batching** | 不等整个 batch 完成就加新请求 | 吞吐 5-10x | 无 |
| **Tensor Parallelism** | 把一层切开分给多个 GPU | N×GPU 数 | 无 |

**投机解码**特别有意思，我们下一个 Part 专门讲。

## 小结

1. ✅ 推理慢的根源：自回归生成时重复计算之前 token 的 K、V → O(N²)
2. ✅ **KV Cache**：把算过的 K、V 存起来 → O(N²) 降到 O(N)
3. ✅ KV Cache 的内存问题：长序列 + 多并发 → 显存爆炸
4. ✅ **KV Cache 量化**：用 INT8/INT4 存储 K、V → 内存再省 2~4 倍
5. ✅ **vLLM / PagedAttention**：把 KV Cache 分页管理 → 消除内存碎片 → 吞吐 2-4x
6. ✅ **FlashAttention**：在 SRAM 里算完，中间结果不写显存 → 快 2-4x
7. ✅ Prefill（计算瓶颈）vs Decode（显存带宽瓶颈）
8. ✅ 第一个 token 慢（prefill），后面的快（decode + KV Cache）

**一句话总结**：LLM 推理慢是因为重复计算 → KV Cache 存起来 → 但存太多内存爆了 → 量化 + 分页管理 → FlashAttention 让每次算得更快。